# 🍽️ Cantina Escolar — Data Warehouse, Análise de Dados e Big Data

**Projeto Integrador — Grupo 3 (Data Science)**

Este notebook parte dos **dados já limpos** (camada **Silver**) da cantina, monta o nosso
**Data Warehouse (DW) em modelo estrela** e, a partir dele, gera as tabelas de análise (camada **Gold**):

```text
Bronze (dados brutos) ──▶ Silver (dados limpos) ──▶ DW (modelo estrela) ──▶ Gold (respostas prontas)
```

| Etapa | O que faz | Conteúdo da disciplina usado |
|---|---|---|
| 1. Carregar | Lê as 8 tabelas limpas e confere se as colunas estão certas | — |
| 2. Juntar | Cruza vendas com cardápio, alunos, calendário e clima (base dos modelos) | — |
| 3. **Data Warehouse** | Monta o **modelo estrela**: tabelas **fato** (o que aconteceu) ligadas a tabelas **dimensão** (quem, o quê, quando, como) | Modelagem dimensional |
| 4. Camada Gold | Gera as 5 tabelas pedidas na especificação (+ inflação) **com consultas SQL no DW** | — |
| 5. Hadoop | **MapReduce** (contagem distribuída), **Sqoop** (exportação para banco) e **Hive** (consultas SQL na estrela) | Aula 08 |
| 6. Regressão | Prevê **quantos itens a cantina vai vender** em um dia, e sugere a produção | Aula 06 |
| 7. Classificação | Prevê se uma **compra fiado vai atrasar** o pagamento | Aula 05 |
| 8. Spark | **Spark SQL** e **Structured Streaming** simulando a chegada das vendas dia a dia | Aula 09 |
| 9. Resumo | Principais números e exportação dos resultados | — |

> **Como usar:** `Ambiente de execução → Executar tudo`. A única coisa a ajustar é a célula de
> **configuração** (seção 1), dizendo onde estão os dados limpos.

**Sobre os gráficos:** foram pensados para quem **não é da área**. Cada gráfico tem um título em forma
de pergunta, os valores escritos nas barras e uma linha **💡 Como ler** logo abaixo. Usamos apenas
gráficos de **colunas/barras** e de **linhas**, e cada categoria (produto, turma, forma de pagamento...)
tem **sempre a mesma cor** em todo o notebook.

**O que ficou de fora das aulas (e por quê):** as simulações de **YARN**, **Pig** e **HBase** da Aula 08
não acrescentam nada a este projeto — o volume de dados da cantina cabe na memória, e a mesma ideia de
"fluxo de transformação" (Pig) e "consulta" (Hive) já está coberta pelo pandas, pelo SQL e pelo Spark.
A coleta de uma API externa da Aula 09 também não se aplica: aqui os "dados ao vivo" são as próprias
vendas da cantina.

## 0. Instalação e bibliotecas

O Colab já vem com quase tudo. Só garantimos o **PySpark** (usado na seção 7).

In [ ]:
%pip -q install pyspark

In [ ]:
import os
import re
import json
import shutil
import sqlite3
import subprocess
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.patches import Patch, FancyBboxPatch
import seaborn as sns
from IPython.display import display, Markdown

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
                             accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

# ------------------------- Visual dos gráficos -------------------------
# Paleta categórica em ORDEM FIXA (testada para daltonismo): a 1ª categoria é sempre azul,
# a 2ª sempre laranja, e assim por diante. Assim a mesma coisa tem a mesma cor em todo o notebook.
CORES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
COR = CORES[0]
COR_DESTAQUE = CORES[1]       # usada para chamar atenção para UMA barra (ex.: o horário de pico)
COR_NEUTRA = "#c9c8c2"        # barras "de fundo", sem destaque
COR_TEXTO, COR_TEXTO_2 = "#0b0b0b", "#52514e"
# Semáforo (só para situação de pagamento e acerto/erro): verde = bom, amarelo = atenção, vermelho = ruim
COR_STATUS = {"Quitado": "#0ca30c", "Em aberto": "#fab219", "Atrasado": "#d03b3b"}
COR_OK, COR_ERRO = "#0ca30c", "#d03b3b"

sns.set_theme(style="whitegrid")
plt.rcParams.update({
    "figure.figsize": (11, 5), "figure.dpi": 110,
    "font.size": 11, "axes.titlesize": 14, "axes.titleweight": "bold", "axes.titlepad": 12,
    "axes.titlelocation": "left", "axes.labelsize": 11, "axes.labelcolor": COR_TEXTO_2,
    "axes.edgecolor": "#d6d5cf", "axes.spines.top": False, "axes.spines.right": False,
    "xtick.color": COR_TEXTO_2, "ytick.color": COR_TEXTO_2,
    "grid.color": "#ecebe6", "grid.linewidth": 0.8,
    "legend.frameon": False, "legend.fontsize": 10,
})

DIAS_SEMANA = {0: "Seg", 1: "Ter", 2: "Qua", 3: "Qui", 4: "Sex", 5: "Sáb", 6: "Dom"}
ORDEM_DIAS = list(DIAS_SEMANA.values())
COR_DIA = {dia: CORES[i] for i, dia in enumerate(ORDEM_DIAS)}
NOMES_MES = {1: "Jan", 2: "Fev", 3: "Mar", 4: "Abr", 5: "Mai", 6: "Jun",
             7: "Jul", 8: "Ago", 9: "Set", 10: "Out", 11: "Nov", 12: "Dez"}
SEGMENTOS = ["Fundamental I", "Fundamental II", "Médio"]
COR_SEGMENTO = dict(zip(SEGMENTOS, CORES))
FAIXAS_TEMP = [-np.inf, 17, 20, 23, np.inf]
ROTULOS_TEMP = ["Frio (<17°C)", "Ameno (17-20°C)", "Agradável (20-23°C)", "Quente (>23°C)"]
COR_TEMP = dict(zip(ROTULOS_TEMP, ["#2a78d6", "#86b6ef", "#f2a07a", "#d95926"]))  # frio = azul, quente = laranja


def br(valor, casas=0):
    """Número no formato brasileiro: 1234.5 -> '1.234,5'."""
    texto = f"{valor:,.{casas}f}"
    return texto.replace(",", "X").replace(".", ",").replace("X", ".")


def reais(valor, casas=0):
    return f"R$ {br(valor, casas)}"


def nome_mes(ano_mes):
    """'2026-10' -> 'Out/2026'."""
    ano, mes = str(ano_mes).split("-")[:2]
    return f"{NOMES_MES[int(mes)]}/{ano}"


def como_ler(texto):
    display(Markdown(f"> 💡 **Como ler:** {texto}"))


def escrever_valores(ax, formato=br, horizontal=False):
    """Escreve o valor de cada barra na ponta dela (o leitor não precisa adivinhar pela régua)."""
    for barra in ax.patches:
        valor = barra.get_width() if horizontal else barra.get_height()
        if not np.isfinite(valor) or valor == 0:
            continue
        if horizontal:
            ax.annotate(formato(valor), (barra.get_x() + valor, barra.get_y() + barra.get_height() / 2),
                        xytext=(4, 0), textcoords="offset points", ha="left", va="center",
                        fontsize=9.5, color=COR_TEXTO_2)
        else:
            ax.annotate(formato(valor), (barra.get_x() + barra.get_width() / 2, barra.get_y() + valor),
                        xytext=(0, 3), textcoords="offset points", ha="center", va="bottom",
                        fontsize=9.5, color=COR_TEXTO_2)
    if horizontal:
        ax.set_xlim(0, ax.get_xlim()[1] * 1.15)
    else:
        ax.set_ylim(0, ax.get_ylim()[1] * 1.12)


def colunas(ax, nomes, valores, cores, titulo, ylabel="", formato=br, rotacao=0):
    """Gráfico de colunas (vertical) com cores, valores escritos e grade só na horizontal."""
    ax.bar([str(n) for n in nomes], valores, color=cores, width=0.65, edgecolor="white", linewidth=2)
    ax.set_title(titulo)
    ax.set_ylabel(ylabel)
    ax.set_xlabel("")
    ax.grid(axis="x", visible=False)
    ax.tick_params(axis="x", rotation=rotacao)
    escrever_valores(ax, formato)


def barras(ax, nomes, valores, cores, titulo, xlabel="", formato=br):
    """Barras deitadas (bom para nomes compridos). A primeira da lista fica no topo."""
    ax.barh([str(n) for n in nomes], valores, color=cores, height=0.7, edgecolor="white", linewidth=2)
    ax.invert_yaxis()
    ax.set_title(titulo)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("")
    ax.grid(axis="y", visible=False)
    escrever_valores(ax, formato, horizontal=True)


def eixo_dias(ax, rotulos, maximo=15):
    """Eixo X com um dia por ponto (sem "buracos" entre datas), mostrando no máximo ~15 rótulos."""
    rotulos = list(rotulos)
    passo = max(1, int(np.ceil(len(rotulos) / maximo)))
    posicoes = list(range(0, len(rotulos), passo))
    ax.set_xticks(posicoes, [rotulos[i] for i in posicoes])
    ax.set_xlim(-0.5, len(rotulos) - 0.5)


def legenda(ax, mapa_cores, titulo=None, **kwargs):
    ax.legend(handles=[Patch(color=c, label=n) for n, c in mapa_cores.items()], title=titulo, **kwargs)


def eixo_reais(ax, eixo="y"):
    fmt = mticker.FuncFormatter(lambda v, _: reais(v))
    (ax.yaxis if eixo == "y" else ax.xaxis).set_major_formatter(fmt)


print("Bibliotecas carregadas.")

## 1. Configuração e carregamento dos dados limpos

Os dados limpos (camada **Silver**) devem estar em uma pasta com **um arquivo por tabela**.
Cada arquivo pode ser `.csv`, `.parquet` ou `.xlsx`, com estes nomes e colunas:

| Arquivo | Colunas obrigatórias |
|---|---|
| `vendas` | id_venda, data_hora, id_aluno, id_produto, quantidade, valor_unitario, valor_total, forma_pagamento |
| `devedores` | id_devedor, id_aluno, data_divida, valor_devido, status, data_quitacao |
| `cardapio` | id_produto, nome_produto, categoria, preco_venda, custo_unitario |
| `perdas` | id_perda, data, id_produto, quantidade, motivo, valor_prejuizo |
| `calendario` | data, tipo_dia, descricao, turno_letivo |
| `clima` | data, temperatura_media, temperatura_max, condicao, precipitacao_mm |
| `alunos` | turma, ano_letivo, turno, id_aluno, nome, data_nascimento, restricao_alimentar |
| `inflacao` | mes_referencia, indice_inflacao, categoria_produto |

O que se espera da limpeza (feita em outra etapa): **IDs no mesmo formato em todas as tabelas**
(`id_aluno` e `id_produto` numéricos), datas em `aaaa-mm-dd`, sem duplicatas, sem valores negativos
ou absurdos, `status` em *Quitado / Em aberto / Atrasado* e categorias da inflação com os mesmos nomes
do cardápio. Colunas extras são permitidas e simplesmente ignoradas.

**Escolha a origem** na célula abaixo:
- `"github"` → baixa o repositório do projeto e lê `dados/silver/` (padrão);
- `"drive"` → lê de uma pasta do seu Google Drive.

In [ ]:
# ===================== CONFIGURAÇÃO (ajuste se necessário) =====================
ORIGEM_DADOS = "github"          # "github" ou "drive"

REPO_URL = "https://github.com/PedroFreitasDev/P.I-Data-Science.git"
PASTA_REPO = "/content/P.I-Data-Science"
SUBPASTA_LIMPOS = "dados/silver"  # pasta dos dados limpos dentro do repositório

PASTA_DRIVE = "/content/drive/MyDrive/P.I-Data-Science/dados/silver"

PASTA_SAIDA = "/content/resultados"   # onde as tabelas Gold e o banco serão salvos
# ===============================================================================

if ORIGEM_DADOS == "github":
    if os.path.isdir(os.path.join(PASTA_REPO, ".git")):
        subprocess.run(["git", "-C", PASTA_REPO, "pull", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL, PASTA_REPO], check=True)
    PASTA_DADOS = os.path.join(PASTA_REPO, SUBPASTA_LIMPOS)
elif ORIGEM_DADOS == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    PASTA_DADOS = PASTA_DRIVE
else:
    raise ValueError('ORIGEM_DADOS deve ser "github" ou "drive"')

os.makedirs(PASTA_SAIDA, exist_ok=True)
print("Lendo os dados limpos de:", PASTA_DADOS)

In [ ]:
# "Contrato" das tabelas limpas: nome do arquivo -> colunas obrigatórias
CONTRATO = {
    "vendas":     ["id_venda", "data_hora", "id_aluno", "id_produto", "quantidade",
                   "valor_unitario", "valor_total", "forma_pagamento"],
    "devedores":  ["id_devedor", "id_aluno", "data_divida", "valor_devido", "status", "data_quitacao"],
    "cardapio":   ["id_produto", "nome_produto", "categoria", "preco_venda", "custo_unitario"],
    "perdas":     ["id_perda", "data", "id_produto", "quantidade", "motivo", "valor_prejuizo"],
    "calendario": ["data", "tipo_dia", "descricao", "turno_letivo"],
    "clima":      ["data", "temperatura_media", "temperatura_max", "condicao", "precipitacao_mm"],
    "alunos":     ["turma", "ano_letivo", "turno", "id_aluno", "nome", "data_nascimento",
                   "restricao_alimentar"],
    "inflacao":   ["mes_referencia", "indice_inflacao", "categoria_produto"],
}
LEITORES = {".parquet": pd.read_parquet, ".csv": pd.read_csv, ".xlsx": pd.read_excel}


def carregar(nome):
    # Procura o arquivo da tabela em qualquer um dos formatos aceitos
    for extensao, leitor in LEITORES.items():
        caminho = os.path.join(PASTA_DADOS, nome + extensao)
        if os.path.exists(caminho):
            df = leitor(caminho)
            faltando = [c for c in CONTRATO[nome] if c not in df.columns]
            if faltando:
                raise ValueError(f"'{nome}{extensao}' está sem as colunas obrigatórias: {faltando}")
            return df
    raise FileNotFoundError(f"Não encontrei '{nome}' (.parquet, .csv ou .xlsx) em {PASTA_DADOS}")


dados = {nome: carregar(nome) for nome in CONTRATO}
display(pd.DataFrame({"tabela": list(dados),
                      "linhas": [len(d) for d in dados.values()],
                      "colunas": [d.shape[1] for d in dados.values()]}))

### 1.1 Acertando os tipos

Mesmo com os dados limpos, ao ler um CSV as datas chegam como texto. Aqui convertemos datas,
e criamos colunas auxiliares nos alunos: **série** (1 a 12), **segmento** (Fundamental I, Fundamental II,
Médio) e **idade**. Também fixamos **uma cor para cada categoria de produto**, usada em todos os gráficos.

In [ ]:
vendas = dados["vendas"].copy()
devedores = dados["devedores"].copy()
cardapio = dados["cardapio"].copy()
perdas = dados["perdas"].copy()
calendario = dados["calendario"].copy()
clima = dados["clima"].copy()
alunos = dados["alunos"].copy()
inflacao = dados["inflacao"].copy()

vendas["data_hora"] = pd.to_datetime(vendas["data_hora"])
devedores["data_divida"] = pd.to_datetime(devedores["data_divida"])
devedores["data_quitacao"] = pd.to_datetime(devedores["data_quitacao"], errors="coerce")
perdas["data"] = pd.to_datetime(perdas["data"])
calendario["data"] = pd.to_datetime(calendario["data"])
clima["data"] = pd.to_datetime(clima["data"])
alunos["data_nascimento"] = pd.to_datetime(alunos["data_nascimento"])
inflacao["mes_referencia"] = pd.to_datetime(inflacao["mes_referencia"].astype(str)).dt.to_period("M")

# O cardápio pode ter mais de uma versão (1º e 2º semestre): fica a mais recente de cada produto
cardapio = cardapio.drop_duplicates("id_produto", keep="last")


def serie_da_turma(turma):
    # "7º ano A" -> 7 ; "2ª série EM B" -> 11 (Ensino Médio vem depois do 9º ano)
    numero = int(re.match(r"\s*(\d+)", str(turma)).group(1))
    return numero + 9 if "EM" in str(turma) else numero


alunos["serie"] = alunos["turma"].apply(serie_da_turma)
alunos["segmento"] = pd.cut(alunos["serie"], [0, 5, 9, 12],
                            labels=["Fundamental I", "Fundamental II", "Médio"]).astype(str)
alunos["idade"] = alunos["ano_letivo"] - alunos["data_nascimento"].dt.year

# Ordem "natural" das turmas (1º ano A, 1º ano B, ..., 3ª série EM B) para os gráficos
ORDEM_TURMAS = alunos.sort_values(["serie", "turma"])["turma"].unique().tolist()

# Uma cor fixa por categoria de produto (a mesma em todos os gráficos)
CATEGORIAS = sorted(cardapio["categoria"].dropna().unique())
COR_CATEGORIA = {c: CORES[i % len(CORES)] for i, c in enumerate(CATEGORIAS)}

print(f"Período das vendas: {vendas['data_hora'].min():%d/%m/%Y} a {vendas['data_hora'].max():%d/%m/%Y}")
print(f"{alunos['id_aluno'].nunique()} alunos em {len(ORDEM_TURMAS)} turmas, "
      f"{cardapio['id_produto'].nunique()} produtos no cardápio")

## 2. Juntando as tabelas (enriquecimento)

Uma venda sozinha diz pouco: "aluno 1234 comprou o produto 7". Juntando com as outras tabelas,
ela passa a dizer **"um aluno do 7º ano A comprou um misto quente (Lanches Naturais) numa
terça-feira fria e chuvosa"**.

Aqui montamos as tabelas "largas" usadas pelos **modelos de Machine Learning** (seções 6 e 7) e pelo
**Spark** (seção 8): vendas enriquecidas (`ve`), um registro por dia de aula (`dias`), fiado (`dev`) e
perdas (`pe`). As **análises de negócio** (camada Gold) vêm do **Data Warehouse**, na seção 3.

> Os IDs de alunos recomeçam a cada ano letivo, por isso a junção com alunos usa **id_aluno + ano**.

In [ ]:
vendas["data"] = vendas["data_hora"].dt.normalize()
vendas["ano_letivo"] = vendas["data_hora"].dt.year
vendas["mes"] = vendas["data_hora"].dt.to_period("M")
vendas["hora"] = vendas["data_hora"].dt.hour
vendas["dia_semana"] = vendas["data"].dt.dayofweek.map(DIAS_SEMANA)

ve = (vendas
      .merge(cardapio[["id_produto", "nome_produto", "categoria", "custo_unitario"]],
             on="id_produto", how="left")
      .merge(alunos[["id_aluno", "ano_letivo", "turma", "serie", "segmento", "turno"]],
             on=["id_aluno", "ano_letivo"], how="left")
      .merge(clima, on="data", how="left"))
ve["lucro"] = ve["valor_total"] - ve["quantidade"] * ve["custo_unitario"]

# Um registro por dia com vendas
vendas_dia = (ve.groupby("data")
                .agg(n_vendas=("id_venda", "nunique"),
                     itens_vendidos=("quantidade", "sum"),
                     faturamento=("valor_total", "sum"))
                .reset_index())

dias = (vendas_dia
        .merge(calendario, on="data", how="left")
        .merge(clima, on="data", how="left"))
dias["dia_semana"] = dias["data"].dt.dayofweek.map(DIAS_SEMANA)
dias["mes"] = dias["data"].dt.month
dias["dia_do_mes"] = dias["data"].dt.day

print(f"Vendas enriquecidas: {len(ve):,} linhas | Dias com vendas: {len(dias)}")
sem_turma = ve["turma"].isna().mean()
sem_produto = ve["categoria"].isna().mean()
print(f"Vendas sem turma encontrada: {sem_turma:.1%} | sem produto no cardápio: {sem_produto:.1%}")
display(ve.head())

# Fiado (devedores) com os dados do aluno
STATUS_ATRASADO, STATUS_ABERTO, STATUS_QUITADO = "Atrasado", "Em aberto", "Quitado"
dev = devedores.copy()
dev["ano_letivo"] = dev["data_divida"].dt.year
dev["mes"] = dev["data_divida"].dt.to_period("M")
dev = dev.merge(alunos[["id_aluno", "ano_letivo", "turma", "serie", "segmento", "turno",
                        "idade", "restricao_alimentar"]],
                on=["id_aluno", "ano_letivo"], how="left")
dev["valor_atrasado"] = dev["valor_devido"].where(dev["status"] == STATUS_ATRASADO, 0)
dev["valor_em_aberto"] = dev["valor_devido"].where(dev["status"] == STATUS_ABERTO, 0)

# Perdas com o nome e a categoria do produto
pe = perdas.merge(cardapio[["id_produto", "nome_produto", "categoria"]], on="id_produto", how="left")

### 2.1 Primeira olhada (análise exploratória)

In [ ]:
print("Resumo dos dias de aula:")
display(dias[["n_vendas", "itens_vendidos", "faturamento", "temperatura_media", "precipitacao_mm"]]
        .describe().T)

# ---- Faturamento dia a dia (linha) ----
media_dia = dias["faturamento"].mean()
fig, ax = plt.subplots(figsize=(12, 4.5))
rotulos_dias = dias["data"].dt.strftime("%d/%m")
ax.plot(np.arange(len(dias)), dias["faturamento"], color=COR, linewidth=2, marker="o", markersize=6,
        label="Faturamento do dia")
ax.axhline(media_dia, color=COR_DESTAQUE, linewidth=2, linestyle="--",
           label=f"Média: {reais(media_dia)} por dia")
ax.set_title("Quanto a cantina faturou em cada dia de aula?")
eixo_reais(ax)
eixo_dias(ax, rotulos_dias)
ax.legend(loc="upper left", bbox_to_anchor=(0, -0.08), ncol=2)
plt.tight_layout()
plt.show()
como_ler("cada ponto azul é um dia de aula. Pontos acima da linha laranja tracejada foram dias "
         "**melhores que a média**; abaixo, dias mais fracos.")

# ---- Forma de pagamento (colunas) e hora do dia (colunas com destaque) ----
pagamento = ve.groupby("forma_pagamento")["valor_total"].sum().sort_values(ascending=False)
COR_PAGAMENTO = {forma: CORES[i] for i, forma in enumerate(sorted(pagamento.index))}
por_hora = ve.groupby("hora")["id_venda"].nunique()

fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
colunas(axes[0], pagamento.index, pagamento.values, [COR_PAGAMENTO[f] for f in pagamento.index],
        "Como os alunos pagam?", "faturamento (R$)", formato=reais, rotacao=15)
eixo_reais(axes[0])
colunas(axes[1], [f"{h}h" for h in por_hora.index], por_hora.values,
        [COR_DESTAQUE if h == por_hora.idxmax() else COR_NEUTRA for h in por_hora.index],
        "Em que horário a cantina vende mais?", "número de vendas")
plt.tight_layout()
plt.show()
como_ler("à esquerda, quanto entrou em reais por cada forma de pagamento (barra mais alta = mais usada). "
         f"À direita, quantas vendas aconteceram em cada hora; a barra **laranja** é o horário de pico "
         f"({por_hora.idxmax()}h).")

## 3. Data Warehouse (DW) — o modelo estrela ⭐

Um **Data Warehouse** é um banco de dados organizado **para responder perguntas**, e não para registrar
o dia a dia da cantina. O jeito mais usado de organizá-lo é o **modelo estrela**:

- **Tabela fato (centro da estrela)** → guarda **o que aconteceu**, com os **números** que queremos
  somar: cada item vendido, com quantidade, valor e lucro.
- **Tabelas dimensão (pontas da estrela)** → guardam **o contexto** do fato, as perguntas que fazemos sobre ele:

| Pergunta | Dimensão | Exemplos de atributos |
|---|---|---|
| **Quando?** | `dim_tempo` | dia, mês, semestre, dia da semana, tipo de dia (letivo, feriado...) |
| **Quem comprou?** | `dim_aluno` | nome, turma, série, segmento, turno, restrição alimentar |
| **O quê?** | `dim_produto` | nome do produto, categoria, preço, custo |
| **Como pagou?** | `dim_pagamento` | forma de pagamento, à vista ou a prazo |
| **Com que tempo?** | `dim_clima` | temperatura, chuva, condição do céu, faixa de temperatura |

A fato só guarda **números de identificação (chaves `sk_...`)** que apontam para cada dimensão.
Para responder "quanto o 7º ano A comprou de Salgados em dias de chuva?", basta **ligar (JOIN)** a fato
às dimensões que interessam — sempre da mesma forma, por mais diferente que seja a pergunta.

**Detalhes do nosso modelo:**
- **Grão** (o que é uma linha da `fato_vendas`): **um produto vendido em uma venda**.
- **Chaves substitutas (`sk_`)**: números criados pelo DW, independentes da origem. São necessárias
  porque o `id_aluno` se repete a cada ano letivo — no DW, cada aluno-ano tem a sua própria `sk_aluno`.
- **Registro "Não identificado" (chave 0)**: se uma venda chega com um aluno ou produto que não existe
  no cadastro, ela não é perdida — aponta para a linha 0 da dimensão.
- **Mais duas fatos usando as MESMAS dimensões** (`fato_perdas` e `fato_fiado`). Isso se chama
  **dimensões conformadas**: como o produto é o mesmo em vendas e em perdas, dá para comparar
  "quanto vendemos" com "quanto jogamos fora" de cada produto.
- A tabela de **inflação** fica fora da estrela: ela é mensal e por categoria (não por venda),
  e é usada só como referência no bônus 4.6.

### 3.1 Construindo as dimensões e as fatos

In [ ]:
def chave_data(datas):
    # 2026-10-06 -> 20261006: chave da dimensão tempo (padrão em DWs, fácil de ler)
    return pd.to_datetime(pd.Series(datas)).dt.strftime("%Y%m%d").astype(int).values


def nao_identificado(dimensao, sk, **valores):
    # Linha 0 de cada dimensão: recebe os fatos cujo aluno/produto/dia não foi encontrado
    resultado = pd.concat([pd.DataFrame([{sk: 0, **valores}]), dimensao], ignore_index=True)
    for coluna in dimensao.columns:   # mantém números como números (o vazio vira NULL no banco)
        if pd.api.types.is_integer_dtype(dimensao[coluna]):
            resultado[coluna] = resultado[coluna].astype("Int64")
        elif pd.api.types.is_float_dtype(dimensao[coluna]):
            resultado[coluna] = resultado[coluna].astype(float)
    return resultado[dimensao.columns]


def buscar_chave(fatos, dimensao, sk, chaves_naturais):
    # Troca a chave da ORIGEM (ex.: id_produto) pela chave substituta da dimensão (ex.: sk_produto)
    ref = dimensao.loc[dimensao[sk] != 0, chaves_naturais + [sk]]
    resultado = fatos.merge(ref, on=chaves_naturais, how="left")
    resultado[sk] = resultado[sk].fillna(0).astype(int)
    return resultado


# ---------------- dim_tempo: QUANDO? (um registro por dia, com ou sem aula) ----------------
todas_as_datas = (pd.concat([calendario["data"], clima["data"], vendas["data"],
                             perdas["data"], devedores["data_divida"]])
                    .dt.normalize().drop_duplicates().sort_values())
dim_tempo = (pd.DataFrame({"data": todas_as_datas})
               .merge(calendario[["data", "tipo_dia", "descricao", "turno_letivo"]]
                      .drop_duplicates("data"), on="data", how="left"))
dim_tempo.insert(0, "sk_tempo", chave_data(dim_tempo["data"]))
dim_tempo["dia"] = dim_tempo["data"].dt.day
dim_tempo["mes"] = dim_tempo["data"].dt.month
dim_tempo["nome_mes"] = dim_tempo["mes"].map(NOMES_MES)
dim_tempo["ano"] = dim_tempo["data"].dt.year
dim_tempo["ano_mes"] = dim_tempo["data"].dt.strftime("%Y-%m")
dim_tempo["trimestre"] = dim_tempo["data"].dt.quarter
dim_tempo["semestre"] = np.where(dim_tempo["mes"] <= 6, "1º semestre", "2º semestre")
dim_tempo["dia_semana"] = dim_tempo["data"].dt.dayofweek.map(DIAS_SEMANA)
dim_tempo["num_dia_semana"] = dim_tempo["data"].dt.dayofweek + 1
dim_tempo["fim_de_semana"] = np.where(dim_tempo["num_dia_semana"] >= 6, "Sim", "Não")
dim_tempo[["tipo_dia", "descricao", "turno_letivo"]] = (dim_tempo[["tipo_dia", "descricao", "turno_letivo"]]
                                                        .fillna("não informado"))

# ---------------- dim_clima: COM QUE TEMPO? (um registro por dia com medição) ----------------
dim_clima = (clima.drop_duplicates("data").sort_values("data").reset_index(drop=True)
                  [["data", "temperatura_media", "temperatura_max", "condicao", "precipitacao_mm"]])
dim_clima.insert(0, "sk_clima", np.arange(1, len(dim_clima) + 1))
dim_clima["faixa_temperatura"] = pd.cut(dim_clima["temperatura_media"], FAIXAS_TEMP,
                                        labels=ROTULOS_TEMP).astype(str)
dim_clima["choveu"] = np.where(dim_clima["precipitacao_mm"] > 0, "Sim", "Não")
dim_clima = nao_identificado(dim_clima, "sk_clima", condicao="não informado",
                             faixa_temperatura="não informado", choveu="não informado")

# ---------------- dim_aluno: QUEM COMPROU? (um registro por aluno em cada ano letivo) ----------------
dim_aluno = (alunos.drop_duplicates(["id_aluno", "ano_letivo"])
                   .assign(ordem_turma=lambda d: d["turma"].map({t: i for i, t in enumerate(ORDEM_TURMAS, 1)}))
                   .sort_values(["ano_letivo", "ordem_turma", "id_aluno"]).reset_index(drop=True)
                   [["id_aluno", "ano_letivo", "nome", "turma", "ordem_turma", "serie", "segmento",
                     "turno", "idade", "restricao_alimentar"]])
dim_aluno.insert(0, "sk_aluno", np.arange(1, len(dim_aluno) + 1))
dim_aluno = nao_identificado(dim_aluno, "sk_aluno", nome="Não identificado", turma="Não identificado",
                             segmento="Não identificado", turno="Não identificado",
                             restricao_alimentar="Não identificado", ordem_turma=999)

# ---------------- dim_produto: O QUÊ? (um registro por produto do cardápio) ----------------
dim_produto = (cardapio.sort_values("id_produto").reset_index(drop=True)
                       [["id_produto", "nome_produto", "categoria", "preco_venda", "custo_unitario"]])
dim_produto.insert(0, "sk_produto", np.arange(1, len(dim_produto) + 1))
dim_produto = nao_identificado(dim_produto, "sk_produto", nome_produto="Não identificado",
                               categoria="Não identificado")

# ---------------- dim_pagamento: COMO PAGOU? ----------------
formas = sorted(vendas["forma_pagamento"].dropna().unique())
dim_pagamento = pd.DataFrame({"sk_pagamento": np.arange(1, len(formas) + 1), "forma_pagamento": formas})
dim_pagamento["tipo_pagamento"] = np.where(dim_pagamento["forma_pagamento"].str.lower() == "fiado",
                                           "A prazo (fiado)", "À vista")
dim_pagamento = nao_identificado(dim_pagamento, "sk_pagamento", forma_pagamento="Não identificado",
                                 tipo_pagamento="Não identificado")

# ================= FATO VENDAS (centro da estrela) — grão: 1 produto em 1 venda =================
f = vendas.copy()
f["sk_tempo"] = chave_data(f["data"])
f = buscar_chave(f, dim_aluno, "sk_aluno", ["id_aluno", "ano_letivo"])
f = buscar_chave(f, dim_produto, "sk_produto", ["id_produto"])
f = buscar_chave(f, dim_pagamento, "sk_pagamento", ["forma_pagamento"])
f = buscar_chave(f, dim_clima, "sk_clima", ["data"])
f = f.merge(dim_produto[["sk_produto", "custo_unitario"]], on="sk_produto", how="left")
f["custo_total"] = (f["quantidade"] * f["custo_unitario"]).round(2)
f["lucro"] = (f["valor_total"] - f["custo_total"]).round(2)
fato_vendas = f[["id_venda", "sk_tempo", "sk_aluno", "sk_produto", "sk_pagamento", "sk_clima",
                 "hora", "quantidade", "valor_unitario", "valor_total", "custo_total", "lucro"]]

# ================= FATO PERDAS — grão: 1 registro de perda de 1 produto em 1 dia =================
f = perdas.copy()
f["sk_tempo"] = chave_data(f["data"])
f = buscar_chave(f, dim_produto, "sk_produto", ["id_produto"])
f = buscar_chave(f, dim_clima, "sk_clima", ["data"])
fato_perdas = f[["id_perda", "sk_tempo", "sk_produto", "sk_clima", "motivo", "quantidade", "valor_prejuizo"]]

# ================= FATO FIADO — grão: 1 dívida de 1 aluno =================
f = devedores.copy()
f["ano_letivo"] = f["data_divida"].dt.year
f["sk_tempo"] = chave_data(f["data_divida"])
f = buscar_chave(f, dim_aluno, "sk_aluno", ["id_aluno", "ano_letivo"])
f["valor_atrasado"] = f["valor_devido"].where(f["status"] == STATUS_ATRASADO, 0)
f["valor_em_aberto"] = f["valor_devido"].where(f["status"] == STATUS_ABERTO, 0)
f["valor_quitado"] = f["valor_devido"].where(f["status"] == STATUS_QUITADO, 0)
f["dias_para_pagar"] = (f["data_quitacao"] - f["data_divida"]).dt.days
fato_fiado = f[["id_devedor", "sk_tempo", "sk_aluno", "status", "valor_devido", "valor_atrasado",
                "valor_em_aberto", "valor_quitado", "dias_para_pagar"]]

print("Dimensões e fatos montadas.")
display(fato_vendas.head())

### 3.2 Gravando o DW em um banco de dados (SQLite)

As tabelas são criadas com **chave primária** (`PRIMARY KEY`, identifica cada linha) e
**chaves estrangeiras** (`FOREIGN KEY`, as "pontas" que ligam a fato às dimensões). Com
`PRAGMA foreign_keys = ON`, o próprio banco **recusa** um fato que aponte para uma dimensão inexistente.
O arquivo `cantina_dw.db` pode ser aberto em qualquer ferramenta de BI (Power BI, Metabase, DBeaver...).

In [ ]:
def para_sql(df):
    # Bancos SQL não entendem os tipos "Period", "Categoria" e data do pandas: viram texto
    df = df.copy()
    for coluna in df.columns:
        if isinstance(df[coluna].dtype, (pd.PeriodDtype, pd.CategoricalDtype)):
            df[coluna] = df[coluna].astype(str)
        elif pd.api.types.is_datetime64_any_dtype(df[coluna]):
            df[coluna] = df[coluna].dt.strftime("%Y-%m-%d")
    return df


def tipo_sql(serie):
    if pd.api.types.is_integer_dtype(serie):
        return "INTEGER"
    if pd.api.types.is_float_dtype(serie):
        return "REAL"
    return "TEXT"


# Desenho da estrela: chave primária de cada tabela e, nas fatos, a qual dimensão cada chave aponta
DIMENSOES = {"dim_tempo": dim_tempo, "dim_aluno": dim_aluno, "dim_produto": dim_produto,
             "dim_pagamento": dim_pagamento, "dim_clima": dim_clima}
FATOS = {"fato_vendas": fato_vendas, "fato_perdas": fato_perdas, "fato_fiado": fato_fiado}
CHAVE_PRIMARIA = {"dim_tempo": "sk_tempo", "dim_aluno": "sk_aluno", "dim_produto": "sk_produto",
                  "dim_pagamento": "sk_pagamento", "dim_clima": "sk_clima",
                  "fato_vendas": "id_venda", "fato_perdas": "id_perda", "fato_fiado": "id_devedor"}
LIGACOES = {nome: {c: d for d, c in CHAVE_PRIMARIA.items() if d.startswith("dim_") and c in fato.columns}
            for nome, fato in FATOS.items()}


def criar_tabela(conexao, nome, df):
    linhas = [f"{c} {tipo_sql(df[c])}{' PRIMARY KEY' if c == CHAVE_PRIMARIA[nome] else ''}"
              for c in df.columns]
    linhas += [f"FOREIGN KEY ({c}) REFERENCES {dim}({c})" for c, dim in LIGACOES.get(nome, {}).items()]
    ddl = f"CREATE TABLE {nome} (\n    " + ",\n    ".join(linhas) + "\n)"
    conexao.execute(ddl)
    para_sql(df).to_sql(nome, conexao, if_exists="append", index=False)
    return ddl


CAMINHO_BANCO = os.path.join(PASTA_SAIDA, "cantina_dw.db")
if os.path.exists(CAMINHO_BANCO):
    os.remove(CAMINHO_BANCO)
dw = sqlite3.connect(CAMINHO_BANCO)
dw.execute("PRAGMA foreign_keys = ON")

ddls = {nome: criar_tabela(dw, nome, df) for nome, df in {**DIMENSOES, **FATOS}.items()}  # dimensões primeiro
dw.commit()
print("Comando que criou a tabela fato_vendas (repare nas FOREIGN KEY):\n")
print(ddls["fato_vendas"])


def consultar_dw(sql):
    return pd.read_sql(sql, dw)


# Conferência de integridade: nenhum fato pode apontar para uma dimensão que não existe
checagem = []
for fato, ligacoes in LIGACOES.items():
    for chave, dim in ligacoes.items():
        orfaos = consultar_dw(f"SELECT COUNT(*) AS n FROM {fato} f LEFT JOIN {dim} d ON d.{chave} = f.{chave} "
                              f"WHERE d.{chave} IS NULL")["n"][0]
        sem_cadastro = consultar_dw(f"SELECT COUNT(*) AS n FROM {fato} WHERE {chave} = 0")["n"][0]
        checagem.append((fato, chave, dim, orfaos, sem_cadastro))
print("\nIntegridade das ligações (fatos sem dimensão devem ser 0):")
display(pd.DataFrame(checagem, columns=["fato", "chave", "dimensão", "fatos sem dimensão",
                                        "fatos ligados ao 'Não identificado'"]))

### 3.3 O desenho da estrela

A figura abaixo é gerada **a partir das tabelas reais do DW**: no centro, a `fato_vendas`;
em volta, as cinco dimensões. Cada linha colorida é uma **chave estrangeira** (FK) da fato apontando
para a **chave primária** (PK) da dimensão.

In [ ]:
def desenhar_tabela(ax, x, y_topo, nome, colunas_tab, cor, largura=3.0, altura_linha=0.34, chaves=()):
    """Desenha uma tabela como caixa: cabeçalho colorido + uma linha por coluna. Devolve a posição de cada coluna."""
    altura = altura_linha * (len(colunas_tab) + 1)
    ax.add_patch(FancyBboxPatch((x, y_topo - altura), largura, altura, boxstyle="round,pad=0.02,rounding_size=0.12",
                                facecolor="white", edgecolor=cor, linewidth=2))
    ax.add_patch(FancyBboxPatch((x, y_topo - altura_linha), largura, altura_linha,
                                boxstyle="round,pad=0.02,rounding_size=0.12", facecolor=cor, edgecolor=cor))
    ax.text(x + largura / 2, y_topo - altura_linha / 2, nome, ha="center", va="center",
            color="white", fontsize=11, fontweight="bold")
    posicoes = {}
    for i, coluna in enumerate(colunas_tab):
        y = y_topo - altura_linha * (i + 1.5)
        marca = "PK" if i == 0 else ("FK" if coluna in chaves else "")
        ax.text(x + 0.12, y, marca, ha="left", va="center", fontsize=8, fontweight="bold", color=COR_TEXTO_2)
        ax.text(x + 0.55, y, coluna, ha="left", va="center", fontsize=9.5, color=COR_TEXTO,
                fontweight="bold" if marca else "normal")
        posicoes[coluna] = y
    return posicoes


def colunas_para_mostrar(df, limite=10):
    cols = list(df.columns)
    return cols if len(cols) <= limite else cols[:limite - 1] + [f"... (+{len(cols) - limite + 1} colunas)"]


COR_DIM = {"dim_tempo": CORES[0], "dim_aluno": CORES[1], "dim_produto": CORES[2],
           "dim_pagamento": CORES[6], "dim_clima": CORES[4]}
LUGAR_DIM = {"dim_tempo": (0.3, 10.6), "dim_clima": (0.3, 4.6), "dim_pagamento": (6.0, 2.6),
             "dim_aluno": (11.7, 10.6), "dim_produto": (11.7, 4.6)}
X_FATO, Y_FATO, LARG = 6.0, 7.6, 3.0

fig, ax = plt.subplots(figsize=(15, 10.5))
ax.set_xlim(0, 15)
ax.set_ylim(0, 12.6)
ax.axis("off")

pos_fato = desenhar_tabela(ax, X_FATO, Y_FATO, "fato_vendas", list(fato_vendas.columns), "#0d366b",
                           chaves=LIGACOES["fato_vendas"])
for dim, (x, y) in LUGAR_DIM.items():
    pos_dim = desenhar_tabela(ax, x, y, dim, colunas_para_mostrar(DIMENSOES[dim]), COR_DIM[dim])
    chave = CHAVE_PRIMARIA[dim]
    y_fk, y_pk = pos_fato[chave], pos_dim[chave]
    if x > X_FATO + LARG:            # dimensão à direita
        inicio, fim = (X_FATO + LARG, y_fk), (x, y_pk)
    elif x + LARG < X_FATO:          # dimensão à esquerda
        inicio, fim = (X_FATO, y_fk), (x + LARG, y_pk)
    else:                            # dimensão embaixo
        inicio, fim = (X_FATO + LARG / 2, Y_FATO - 0.34 * (len(fato_vendas.columns) + 1)), (x + LARG / 2, y)
    ax.annotate("", xy=fim, xytext=inicio,
                arrowprops=dict(arrowstyle="-|>", color=COR_DIM[dim], linewidth=2.2, mutation_scale=16))

ax.text(0.3, 12.35, "Modelo estrela do DW da cantina", fontsize=16, fontweight="bold", color=COR_TEXTO)
ax.text(0.3, 11.95, "Centro (azul-escuro) = FATO: o que aconteceu, com os números.   "
                    "Pontas = DIMENSÕES: quando, quem, o quê, como pagou e com que tempo.",
        fontsize=10.5, color=COR_TEXTO_2)
ax.text(0.3, 0.3, "PK = chave primária (identifica a linha)   ·   FK = chave estrangeira (aponta para uma dimensão)   ·   "
                  f"fato_vendas: {len(fato_vendas):,} linhas".replace(",", "."),
        fontsize=10, color=COR_TEXTO_2)
plt.show()
como_ler("comece pelo centro. Cada venda da `fato_vendas` carrega cinco números (sk_...). Siga a seta "
         "da mesma cor para descobrir a que dimensão cada número leva: `sk_aluno` leva ao aluno, à turma e à "
         "série; `sk_tempo` leva ao dia, mês e dia da semana; e assim por diante.")

### 3.4 As três fatos compartilham as mesmas dimensões (matriz de barramento)

A **matriz de barramento** mostra quais dimensões cada fato usa. Como as dimensões são **as mesmas
tabelas** para todas as fatos, um mesmo filtro (por exemplo, "categoria = Salgados" ou "mês = Outubro")
vale para vendas, perdas e fiado ao mesmo tempo.

In [ ]:
GRAOS = {"fato_vendas": "1 produto vendido em 1 venda", "fato_perdas": "1 perda de 1 produto em 1 dia",
         "fato_fiado": "1 dívida de 1 aluno"}
matriz = pd.DataFrame("", index=list(FATOS), columns=list(DIMENSOES))
for fato, ligacoes in LIGACOES.items():
    for dim in ligacoes.values():
        matriz.loc[fato, dim] = "✔"
matriz.insert(0, "grão (1 linha = ...)", pd.Series(GRAOS))
display(matriz)

resumo_dw = pd.DataFrame(
    [(n, "Fato" if n.startswith("fato") else "Dimensão", len(df), df.shape[1]) for n, df in {**FATOS, **DIMENSOES}.items()],
    columns=["tabela", "tipo", "linhas", "colunas"])
display(resumo_dw)

fig, ax = plt.subplots(figsize=(11, 4.2))
cores_tab = ["#0d366b" if t == "Fato" else COR_DIM[n] for n, t in zip(resumo_dw["tabela"], resumo_dw["tipo"])]
barras(ax, resumo_dw["tabela"], resumo_dw["linhas"], cores_tab, "Quantas linhas tem cada tabela do DW?",
       "linhas")
plt.tight_layout()
plt.show()
como_ler("as fatos (azul-escuro) crescem todo dia, a cada venda; as dimensões (coloridas) mudam pouco, "
         "porque descrevem coisas estáveis, como alunos, produtos e dias do calendário.")

## 4. Camada Gold — respostas prontas, calculadas a partir do DW

A especificação do projeto pede estas tabelas agregadas:

1. **Venda por turma/mês**
2. **Inadimplência por período/turma**
3. **Relação clima × volume de vendas**
4. **Desperdício por categoria de produto**
5. **Turmas × dias de pico de venda**

E, como bônus, **reajuste de preços × inflação**. Todas são calculadas com **SQL sobre o modelo estrela**:
partimos de uma **fato** e fazemos `JOIN` só nas **dimensões** que a pergunta precisa. Os comentários em
cada consulta mostram qual pergunta cada dimensão responde. As tabelas ficam no dicionário `gold` e são
exportadas no final.

In [ ]:
gold = {}

# ---------------- 4.1 Venda por turma/mês ----------------
# fato_vendas + dim_aluno (QUEM comprou) + dim_tempo (QUANDO)
gold["venda_turma_mes"] = consultar_dw("""
    SELECT a.turma, a.segmento, t.ano_mes AS mes,
           COUNT(DISTINCT f.id_venda)   AS n_vendas,
           SUM(f.quantidade)            AS itens,
           ROUND(SUM(f.valor_total), 2) AS faturamento,
           ROUND(SUM(f.lucro), 2)       AS lucro
    FROM fato_vendas f
    JOIN dim_aluno a ON a.sk_aluno = f.sk_aluno      -- quem comprou
    JOIN dim_tempo t ON t.sk_tempo = f.sk_tempo      -- quando
    WHERE a.sk_aluno <> 0
    GROUP BY a.ordem_turma, a.turma, a.segmento, t.ano_mes
    ORDER BY a.ordem_turma, t.ano_mes
""")
gold["venda_turma_mes"]["ticket_medio"] = (gold["venda_turma_mes"]["faturamento"]
                                           / gold["venda_turma_mes"]["n_vendas"])
display(gold["venda_turma_mes"].head(10))

# Gráfico 1: faturamento por mês, uma coluna para cada segmento
seg_mes = (gold["venda_turma_mes"].groupby(["mes", "segmento"])["faturamento"].sum()
           .unstack().reindex(columns=[s for s in SEGMENTOS if s in set(gold["venda_turma_mes"]["segmento"])]))
fig, ax = plt.subplots(figsize=(11, 4.8))
seg_mes.rename(index=nome_mes).plot.bar(ax=ax, color=[COR_SEGMENTO[s] for s in seg_mes.columns], width=0.75,
                                        edgecolor="white", linewidth=2, rot=0)
ax.set_title("Quanto cada segmento comprou em cada mês?")
ax.set_xlabel("")
ax.grid(axis="x", visible=False)
eixo_reais(ax)
escrever_valores(ax, reais)
ax.legend(title="Segmento")
plt.tight_layout()
plt.show()
como_ler("cada grupo de colunas é um mês; cada cor é um segmento da escola. Quanto mais alta a coluna, "
         "mais dinheiro aquele segmento gastou na cantina naquele mês.")

# Gráfico 2: faturamento total de cada turma, colorido pelo segmento
por_turma_fat = (gold["venda_turma_mes"].groupby(["turma", "segmento"], sort=False)["faturamento"].sum()
                 .reset_index())
fig, ax = plt.subplots(figsize=(11, 0.32 * len(por_turma_fat) + 1.5))
barras(ax, por_turma_fat["turma"], por_turma_fat["faturamento"],
       [COR_SEGMENTO.get(s, COR_NEUTRA) for s in por_turma_fat["segmento"]],
       "Quanto cada turma comprou no período?", "faturamento (R$)", formato=reais)
eixo_reais(ax, "x")
legenda(ax, {s: COR_SEGMENTO[s] for s in SEGMENTOS if s in set(por_turma_fat["segmento"])},
        "Segmento", loc="upper left", bbox_to_anchor=(1.01, 1))
plt.tight_layout()
plt.show()
como_ler("cada barra é uma turma, da 1ª série (em cima) até o Ensino Médio (embaixo). A cor mostra o "
         "segmento. Barras mais compridas = turmas que mais compram.")

In [ ]:
# ---------------- 4.2 Inadimplência por período/turma ----------------
# fato_fiado + dim_aluno (QUEM deve) + dim_tempo (QUANDO comprou fiado)
gold["inadimplencia_turma_mes"] = consultar_dw("""
    SELECT a.turma, a.segmento, t.ano_mes AS mes,
           COUNT(*)                         AS n_dividas,
           ROUND(SUM(f.valor_devido), 2)    AS valor_total,
           ROUND(SUM(f.valor_atrasado), 2)  AS valor_atrasado,
           ROUND(SUM(f.valor_em_aberto), 2) AS valor_em_aberto,
           ROUND(SUM(f.valor_quitado), 2)   AS valor_quitado
    FROM fato_fiado f
    JOIN dim_aluno a ON a.sk_aluno = f.sk_aluno      -- quem deve
    JOIN dim_tempo t ON t.sk_tempo = f.sk_tempo      -- quando
    WHERE a.sk_aluno <> 0
    GROUP BY a.ordem_turma, a.turma, a.segmento, t.ano_mes
    ORDER BY a.ordem_turma, t.ano_mes
""")
gold["inadimplencia_turma_mes"]["pct_atrasado"] = (100 * gold["inadimplencia_turma_mes"]["valor_atrasado"]
                                                   / gold["inadimplencia_turma_mes"]["valor_total"])
display(gold["inadimplencia_turma_mes"].head(10))

ORDEM_STATUS = [STATUS_QUITADO, STATUS_ABERTO, STATUS_ATRASADO]
COLUNA_STATUS = {STATUS_QUITADO: "valor_quitado", STATUS_ABERTO: "valor_em_aberto",
                 STATUS_ATRASADO: "valor_atrasado"}
por_status = consultar_dw("SELECT status, COUNT(*) AS dividas, SUM(valor_devido) AS valor "
                          "FROM fato_fiado GROUP BY status").set_index("status").reindex(ORDEM_STATUS).fillna(0)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.8), gridspec_kw={"width_ratios": [1, 1.4]})
colunas(axes[0], por_status.index, por_status["valor"], [COR_STATUS[s] for s in por_status.index],
        "Situação de todo o dinheiro vendido fiado", "R$", formato=reais)
mes_status = (gold["inadimplencia_turma_mes"].groupby("mes")[[COLUNA_STATUS[s] for s in ORDEM_STATUS]].sum()
              .set_axis(ORDEM_STATUS, axis=1).rename(index=nome_mes))
mes_status.plot.bar(ax=axes[1], color=[COR_STATUS[s] for s in ORDEM_STATUS], width=0.75,
                    edgecolor="white", linewidth=2, rot=0)
axes[1].set_title("Fiado de cada mês: quanto já foi pago e quanto falta?")
axes[1].set_xlabel("")
axes[1].grid(axis="x", visible=False)
escrever_valores(axes[1], reais)
axes[1].legend(title="Situação")
for a in axes:
    eixo_reais(a)
plt.tight_layout()
plt.show()
como_ler("**verde** = já foi pago, **amarelo** = ainda dentro do prazo, **vermelho** = atrasado. "
         "Quanto maior a parte vermelha, maior o risco de a cantina não receber.")

# Por turma: barras empilhadas (cada pedaço da barra é uma situação)
turma_status = (gold["inadimplencia_turma_mes"].groupby("turma", sort=False)
                [[COLUNA_STATUS[s] for s in ORDEM_STATUS]].sum().set_axis(ORDEM_STATUS, axis=1))
fig, ax = plt.subplots(figsize=(11, 0.34 * len(turma_status) + 1.6))
turma_status.plot.barh(ax=ax, stacked=True, color=[COR_STATUS[s] for s in ORDEM_STATUS], width=0.7,
                       edgecolor="white", linewidth=2)
ax.invert_yaxis()
ax.set_title("Quanto cada turma deve no fiado, e em que situação?")
ax.set_xlabel("R$")
ax.set_ylabel("")
ax.grid(axis="y", visible=False)
eixo_reais(ax, "x")
total_turma = turma_status.sum(axis=1)
for i, total in enumerate(total_turma):
    ax.annotate(reais(total), (total, i), xytext=(4, 0), textcoords="offset points", va="center",
                fontsize=9.5, color=COR_TEXTO_2)
ax.set_xlim(0, total_turma.max() * 1.18)
ax.legend(title="Situação", loc="upper left", bbox_to_anchor=(1.01, 1))
plt.tight_layout()
plt.show()
como_ler("cada barra é uma turma que comprou fiado. O tamanho total é quanto ela comprou fiado; "
         "as cores dividem esse valor em pago (verde), em aberto (amarelo) e atrasado (vermelho).")

In [ ]:
# ---------------- 4.3 Relação clima × volume de vendas ----------------
# fato_vendas + dim_clima (COM QUE TEMPO). A média é por dia: itens do grupo ÷ dias do grupo
gold["clima_vendas"] = consultar_dw("""
    SELECT c.condicao, c.faixa_temperatura,
           COUNT(DISTINCT f.sk_tempo)     AS dias,
           SUM(f.quantidade)              AS itens,
           ROUND(SUM(f.valor_total), 2)   AS faturamento
    FROM fato_vendas f
    JOIN dim_clima c ON c.sk_clima = f.sk_clima      -- com que tempo
    WHERE c.sk_clima <> 0
    GROUP BY c.condicao, c.faixa_temperatura
""")
gold["clima_vendas"]["media_itens"] = gold["clima_vendas"]["itens"] / gold["clima_vendas"]["dias"]
gold["clima_vendas"]["media_faturamento"] = gold["clima_vendas"]["faturamento"] / gold["clima_vendas"]["dias"]
display(gold["clima_vendas"])

print("Correlação com o número de itens vendidos no dia (de -1 a 1; perto de 0 = pouca relação):")
display(dias[["itens_vendidos", "temperatura_media", "temperatura_max", "precipitacao_mm"]]
        .corr()["itens_vendidos"].drop("itens_vendidos").to_frame("correlação"))

# Cada dia tem uma só condição e uma só faixa, então somar itens e dias dá a média exata por dia
por_condicao = gold["clima_vendas"].groupby("condicao")[["itens", "dias"]].sum()
por_condicao = (por_condicao["itens"] / por_condicao["dias"]).sort_values(ascending=False)
COR_CONDICAO = {c: CORES[i] for i, c in enumerate(sorted(por_condicao.index))}
por_faixa = gold["clima_vendas"].groupby("faixa_temperatura")[["itens", "dias"]].sum()
por_faixa = (por_faixa["itens"] / por_faixa["dias"]).reindex([r for r in ROTULOS_TEMP if r in por_faixa.index])

fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
colunas(axes[0], [c.replace("_", " ").capitalize() for c in por_condicao.index], por_condicao.values,
        [COR_CONDICAO[c] for c in por_condicao.index], "Em que tempo a cantina vende mais?",
        "itens vendidos por dia (média)", rotacao=12)
colunas(axes[1], por_faixa.index, por_faixa.values, [COR_TEMP[f] for f in por_faixa.index],
        "Dias frios ou quentes vendem mais?", "itens vendidos por dia (média)", rotacao=12)
plt.tight_layout()
plt.show()
como_ler("cada coluna é a **média de itens vendidos em um dia** com aquele tempo. Usamos a média (e não o "
         "total) porque alguns tipos de tempo acontecem em mais dias que outros.")

# O que muda no "carrinho" conforme a temperatura: fato_vendas + dim_clima + dim_produto
mix = consultar_dw("""
    SELECT c.faixa_temperatura, p.categoria, SUM(f.quantidade) AS itens
    FROM fato_vendas f
    JOIN dim_clima   c ON c.sk_clima   = f.sk_clima     -- com que tempo
    JOIN dim_produto p ON p.sk_produto = f.sk_produto   -- o quê
    WHERE c.sk_clima <> 0 AND p.sk_produto <> 0
    GROUP BY c.faixa_temperatura, p.categoria
""").pivot(index="faixa_temperatura", columns="categoria", values="itens").fillna(0)
mix = 100 * mix.div(mix.sum(axis=1), axis=0)
mix = mix.reindex([r for r in ROTULOS_TEMP if r in mix.index])

fig, ax = plt.subplots(figsize=(12, 4.8))
mix.plot.bar(ax=ax, color=[COR_CATEGORIA.get(c, COR_NEUTRA) for c in mix.columns], width=0.8,
             edgecolor="white", linewidth=2, rot=0)
ax.set_title("O que os alunos compram em dias frios e em dias quentes?")
ax.set_ylabel("% dos itens vendidos")
ax.set_xlabel("")
ax.grid(axis="x", visible=False)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(decimals=0))
ax.legend(title="Categoria", ncol=len(mix.columns), loc="upper center", bbox_to_anchor=(0.5, -0.1))
plt.tight_layout()
plt.show()
como_ler("cada grupo é uma faixa de temperatura; cada cor é uma categoria de produto. A altura mostra que "
         "**fatia** das vendas aquela categoria teve. Se a coluna de Bebidas cresce no calor, por exemplo, os "
         "alunos compram mais bebidas em dias quentes.")

In [ ]:
# ---------------- 4.4 Desperdício por categoria de produto ----------------
# Duas fatos (perdas e vendas) ligadas pela MESMA dim_produto: "drill-across", só possível
# porque as dimensões são compartilhadas (conformadas) entre as fatos.
gold["desperdicio_categoria"] = consultar_dw("""
    WITH perdas AS (
        SELECT p.categoria, COUNT(*) AS registros, SUM(fp.quantidade) AS qtd_perdida,
               ROUND(SUM(fp.valor_prejuizo), 2) AS prejuizo
        FROM fato_perdas fp
        JOIN dim_produto p ON p.sk_produto = fp.sk_produto
        GROUP BY p.categoria),
    vendas AS (
        SELECT p.categoria, ROUND(SUM(fv.valor_total), 2) AS faturamento
        FROM fato_vendas fv
        JOIN dim_produto p ON p.sk_produto = fv.sk_produto
        GROUP BY p.categoria)
    SELECT perdas.*, vendas.faturamento,
           ROUND(100.0 * perdas.prejuizo / vendas.faturamento, 2) AS prejuizo_pct_faturamento
    FROM perdas LEFT JOIN vendas ON vendas.categoria = perdas.categoria
    ORDER BY prejuizo DESC
""")
display(gold["desperdicio_categoria"])

gold["desperdicio_produto_motivo"] = consultar_dw("""
    SELECT p.categoria, p.nome_produto, fp.motivo,
           SUM(fp.quantidade) AS qtd_perdida, ROUND(SUM(fp.valor_prejuizo), 2) AS prejuizo
    FROM fato_perdas fp
    JOIN dim_produto p ON p.sk_produto = fp.sk_produto   -- o quê
    GROUP BY p.categoria, p.nome_produto, fp.motivo
    ORDER BY prejuizo DESC
""")

dc = gold["desperdicio_categoria"]
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
colunas(axes[0], dc["categoria"], dc["prejuizo"], [COR_CATEGORIA.get(c, COR_NEUTRA) for c in dc["categoria"]],
        "Quanto dinheiro foi jogado fora em cada categoria?", "prejuízo (R$)", formato=reais, rotacao=12)
eixo_reais(axes[0])
dc_pct = dc.sort_values("prejuizo_pct_faturamento", ascending=False)
colunas(axes[1], dc_pct["categoria"], dc_pct["prejuizo_pct_faturamento"],
        [COR_CATEGORIA.get(c, COR_NEUTRA) for c in dc_pct["categoria"]],
        "A cada R$ 100 vendidos, quantos reais se perdem?", "reais perdidos a cada R\\$ 100",
        formato=lambda v: reais(v, 2), rotacao=12)
plt.tight_layout()
plt.show()
como_ler("à esquerda, o prejuízo total de cada categoria. À direita, o mesmo prejuízo comparado com o que "
         "a categoria vende: uma categoria pode perder pouco em reais, mas muito em relação ao que vende.")

top_perdas = (gold["desperdicio_produto_motivo"].groupby(["nome_produto", "categoria"])["prejuizo"].sum()
              .nlargest(10).reset_index())
fig, ax = plt.subplots(figsize=(11, 5))
barras(ax, top_perdas["nome_produto"], top_perdas["prejuizo"],
       [COR_CATEGORIA.get(c, COR_NEUTRA) for c in top_perdas["categoria"]],
       "Quais 10 produtos dão mais prejuízo com perdas?", "prejuízo (R$)", formato=lambda v: reais(v, 2))
eixo_reais(ax, "x")
legenda(ax, {c: COR_CATEGORIA[c] for c in CATEGORIAS if c in set(top_perdas["categoria"])}, "Categoria",
        loc="upper left", bbox_to_anchor=(1.01, 1))
plt.tight_layout()
plt.show()
como_ler("o produto do topo é o que mais gera prejuízo. A cor indica a categoria — a mesma cor usada para "
         "ela em todos os outros gráficos.")

In [ ]:
# ---------------- 4.5 Turmas × dias de pico de venda ----------------
# Média de vendas por dia (e não o total), porque nem todo dia da semana aparece o mesmo número de vezes
n_dias_semana = consultar_dw("""
    SELECT t.dia_semana, COUNT(DISTINCT f.sk_tempo) AS n_dias
    FROM fato_vendas f JOIN dim_tempo t ON t.sk_tempo = f.sk_tempo
    GROUP BY t.dia_semana
""").set_index("dia_semana")["n_dias"]

gold["turma_dia_pico"] = consultar_dw("""
    WITH dias_semana AS (
        SELECT t.dia_semana, COUNT(DISTINCT f.sk_tempo) AS n_dias
        FROM fato_vendas f JOIN dim_tempo t ON t.sk_tempo = f.sk_tempo
        GROUP BY t.dia_semana)
    SELECT a.turma, t.dia_semana,
           COUNT(DISTINCT f.id_venda)                   AS vendas,
           1.0 * COUNT(DISTINCT f.id_venda) / d.n_dias  AS media_vendas_por_dia
    FROM fato_vendas f
    JOIN dim_aluno a   ON a.sk_aluno   = f.sk_aluno     -- quem comprou
    JOIN dim_tempo t   ON t.sk_tempo   = f.sk_tempo     -- quando
    JOIN dias_semana d ON d.dia_semana = t.dia_semana
    WHERE a.sk_aluno <> 0
    GROUP BY a.ordem_turma, a.turma, t.dia_semana, d.n_dias
    ORDER BY a.ordem_turma
""")
tdp = gold["turma_dia_pico"]
pico = (tdp.loc[tdp.groupby("turma", sort=False)["media_vendas_por_dia"].idxmax()]
           .rename(columns={"dia_semana": "dia_de_pico"}).set_index("turma")
           .reindex([t for t in ORDEM_TURMAS if t in set(tdp["turma"])]))
print("Dia de pico de cada turma:")
display(pico[["dia_de_pico", "media_vendas_por_dia"]].T)

# Gráfico 1: dia da semana (geral), com o pico em destaque
geral_dia = (consultar_dw("""
    SELECT t.dia_semana, COUNT(DISTINCT f.id_venda) AS vendas
    FROM fato_vendas f JOIN dim_tempo t ON t.sk_tempo = f.sk_tempo
    GROUP BY t.dia_semana
""").set_index("dia_semana")["vendas"] / n_dias_semana).reindex([d for d in ORDEM_DIAS if d in n_dias_semana.index])
fig, ax = plt.subplots(figsize=(10, 4.5))
colunas(ax, geral_dia.index, geral_dia.values,
        [COR_DESTAQUE if d == geral_dia.idxmax() else COR_NEUTRA for d in geral_dia.index],
        "Qual dia da semana tem mais movimento?", "vendas por dia (média)", formato=lambda v: br(v, 1))
plt.tight_layout()
plt.show()
como_ler(f"média de vendas em cada dia da semana. Em **laranja**, o dia de maior movimento ({geral_dia.idxmax()}).")

# Gráfico 2: o dia de pico de cada turma (a cor diz QUAL é o dia)
dias_presentes = [d for d in ORDEM_DIAS if d in set(pico["dia_de_pico"])]
fig, ax = plt.subplots(figsize=(11, 0.32 * len(pico) + 1.6))
barras(ax, pico.index, pico["media_vendas_por_dia"], [COR_DIA[d] for d in pico["dia_de_pico"]],
       "Qual é o dia de pico de cada turma?", "vendas por dia (média) no dia de pico",
       formato=lambda v: br(v, 1))
legenda(ax, {d: COR_DIA[d] for d in dias_presentes}, "Dia de pico", loc="upper left", bbox_to_anchor=(1.01, 1))
plt.tight_layout()
plt.show()
como_ler("cada barra é uma turma. A **cor** mostra em que dia da semana ela mais compra (veja a legenda) e o "
         "**tamanho** mostra quantas vendas, em média, ela faz nesse dia.")

# Gráfico 3: horário de pico (média por dia de aula)
por_hora_dia = consultar_dw("""
    SELECT f.hora, 1.0 * COUNT(DISTINCT f.id_venda) / (SELECT COUNT(DISTINCT sk_tempo) FROM fato_vendas) AS media
    FROM fato_vendas f GROUP BY f.hora ORDER BY f.hora
""").set_index("hora")["media"]
fig, ax = plt.subplots(figsize=(10, 4.5))
colunas(ax, [f"{h}h" for h in por_hora_dia.index], por_hora_dia.values,
        [COR_DESTAQUE if h == por_hora_dia.idxmax() else COR_NEUTRA for h in por_hora_dia.index],
        "Em que horário a fila da cantina é maior?", "vendas por dia (média)", formato=lambda v: br(v, 1))
plt.tight_layout()
plt.show()
como_ler("quantas vendas acontecem, em média, em cada hora de um dia de aula. A coluna **laranja** é o "
         "horário de pico — o momento de ter mais gente no balcão.")

In [ ]:
# ---------------- 4.6 Bônus: reajuste de preços × inflação ----------------
# Inflação acumulada de cada categoria: (1 + i1) x (1 + i2) x ... - 1  (tabela de referência, fora da estrela)
infl = (inflacao.sort_values("mes_referencia")
                .groupby("categoria_produto")["indice_inflacao"]
                .apply(lambda s: (np.prod(1 + s / 100) - 1) * 100)
                .rename("inflacao_acumulada_pct"))
print(f"Inflação publicada de {inflacao['mes_referencia'].min()} a {inflacao['mes_referencia'].max()}")

# Preço praticado de cada produto em cada semestre: fato_vendas + dim_produto + dim_tempo (mediana no pandas)
precos = (consultar_dw("""
    SELECT p.categoria, p.nome_produto, t.semestre, f.valor_unitario
    FROM fato_vendas f
    JOIN dim_produto p ON p.sk_produto = f.sk_produto   -- o quê
    JOIN dim_tempo   t ON t.sk_tempo   = f.sk_tempo     -- quando
    WHERE p.sk_produto <> 0
""").groupby(["categoria", "nome_produto", "semestre"])["valor_unitario"].median().unstack())

if {"1º semestre", "2º semestre"} <= set(precos.columns):
    precos = precos.dropna()
    reajuste = (100 * (precos["2º semestre"] / precos["1º semestre"] - 1)).groupby("categoria").mean()
    gold["reajuste_vs_inflacao"] = (pd.concat([reajuste.rename("reajuste_medio_pct"), infl], axis=1)
                                      .rename_axis("categoria").reset_index())
    gold["reajuste_vs_inflacao"]["diferenca_pp"] = (gold["reajuste_vs_inflacao"]["reajuste_medio_pct"]
                                                    - gold["reajuste_vs_inflacao"]["inflacao_acumulada_pct"])
    display(gold["reajuste_vs_inflacao"])
    print("diferenca_pp negativa = o preço subiu MENOS que a inflação (a margem encolheu).")

    comparacao = (gold["reajuste_vs_inflacao"].set_index("categoria")
                  [["reajuste_medio_pct", "inflacao_acumulada_pct"]]
                  .rename(columns={"reajuste_medio_pct": "Quanto o preço da cantina subiu",
                                   "inflacao_acumulada_pct": "Quanto a inflação subiu"}))
    fig, ax = plt.subplots(figsize=(11, 4.8))
    comparacao.plot.bar(ax=ax, color=[CORES[0], CORES[1]], width=0.75, edgecolor="white", linewidth=2, rot=0)
    ax.set_title("Os preços da cantina acompanharam a inflação?")
    ax.set_ylabel("aumento (%)")
    ax.set_xlabel("")
    ax.grid(axis="x", visible=False)
    escrever_valores(ax, lambda v: f"{br(v, 1)}%")
    plt.tight_layout()
    plt.show()
    como_ler("para cada categoria, a coluna **azul** é quanto a cantina subiu os preços e a **laranja** é "
             "quanto os custos subiram com a inflação. Se a laranja é mais alta, a cantina está ganhando menos "
             "em cada venda.")
else:
    gold["reajuste_vs_inflacao"] = infl.rename_axis("categoria").reset_index()
    print("Ainda não há vendas nos dois semestres; mostrando só a inflação acumulada.")
    display(gold["reajuste_vs_inflacao"])
    ri = gold["reajuste_vs_inflacao"].sort_values("inflacao_acumulada_pct", ascending=False)
    fig, ax = plt.subplots(figsize=(11, 4.5))
    colunas(ax, ri["categoria"], ri["inflacao_acumulada_pct"], [COR_CATEGORIA.get(c, COR_NEUTRA) for c in ri["categoria"]],
            "Quanto os custos de cada categoria subiram com a inflação?", "inflação acumulada (%)",
            formato=lambda v: f"{br(v, 1)}%")
    plt.tight_layout()
    plt.show()
    como_ler("cada coluna mostra quanto ficou mais caro, desde o início do ano, comprar os ingredientes "
             "de cada categoria. Quanto mais alta, mais a cantina precisaria reajustar o preço.")

## 5. Big Data com o ecossistema Hadoop (Aula 08)

Em Big Data, quando os dados não cabem em um computador, o trabalho é dividido entre várias máquinas.
O Hadoop organiza isso. Aqui aplicamos as três peças que fazem sentido para a cantina:

- **MapReduce** — divide uma contagem grande em pedaços (*map*), agrupa os pedaços iguais (*shuffle*)
  e soma tudo no final (*reduce*).
- **Sqoop** — leva tabelas para um banco de dados relacional (aqui, o próprio banco do DW).
- **Hive** — faz perguntas aos dados usando **SQL** — aqui, diretamente no **modelo estrela**.

### 5.1 MapReduce: produtos mais vendidos e faturamento por forma de pagamento

In [ ]:
# Etapa Map: transforma cada venda em um par (chave, valor)
def mapper(registros, chave, valor):
    pares = []
    for registro in registros:
        if pd.notna(registro[chave]):
            pares.append((registro[chave], registro[valor]))
    return pares


# Etapa Shuffle: junta todos os valores de uma mesma chave
def shuffle(pares):
    agrupado = {}
    for chave, valor in pares:
        agrupado.setdefault(chave, []).append(valor)
    return agrupado


# Etapa Reduce: soma os valores de cada chave
def reducer(agrupado):
    return {chave: sum(valores) for chave, valores in agrupado.items()}


def job_mapreduce(df, chave, valor, n_blocos=4):
    # Simula o HDFS: os dados são divididos em blocos, e cada bloco é mapeado separadamente
    registros = df[[chave, valor]].to_dict("records")
    tamanho = int(np.ceil(len(registros) / n_blocos))
    blocos = [registros[i:i + tamanho] for i in range(0, len(registros), tamanho)]
    pares = [par for bloco in blocos for par in mapper(bloco, chave, valor)]
    return reducer(shuffle(pares))


itens_por_produto = job_mapreduce(ve, "nome_produto", "quantidade")
print("[MapReduce] Top 10 produtos por itens vendidos:")
for produto, total in sorted(itens_por_produto.items(), key=lambda x: -x[1])[:10]:
    print(f"  {produto:<32} {total:>7,}")

fat_pagamento = job_mapreduce(ve, "forma_pagamento", "valor_total")
print("\n[MapReduce] Faturamento por forma de pagamento:")
for forma, total in sorted(fat_pagamento.items(), key=lambda x: -x[1]):
    print(f"  {forma:<20} R$ {total:>12,.2f}")

# Conferência: o MapReduce tem que bater com o groupby do pandas
confere = ve.groupby("nome_produto")["quantidade"].sum().to_dict()
print("\nMapReduce confere com o pandas?", confere == itens_por_produto)

### 5.2 Sqoop: exportando as tabelas Gold para o banco do DW

O Sqoop transfere dados entre o Hadoop e bancos relacionais. O DW já está no banco **SQLite**
(`cantina_dw.db`, seção 3.2); aqui levamos para ele também as **tabelas Gold** (prefixo `gold_`).
Assim, quem abrir o banco encontra a estrela completa **e** as respostas prontas lado a lado.

In [ ]:
for nome, tabela in gold.items():
    para_sql(tabela).to_sql(f"gold_{nome}", dw, index=False, if_exists="replace")
dw.commit()

print("[Sqoop] Tabelas no banco", CAMINHO_BANCO)
display(consultar_dw("""
    SELECT name AS tabela,
           CASE WHEN name LIKE 'fato_%' THEN 'Fato (DW)'
                WHEN name LIKE 'dim_%'  THEN 'Dimensão (DW)'
                ELSE 'Gold (resposta pronta)' END AS camada
    FROM sqlite_master WHERE type = 'table' ORDER BY camada, name
"""))

### 5.3 Hive: perguntas em SQL no modelo estrela

O Hive permite consultar dados gigantes com SQL, a linguagem padrão de bancos de dados.
Repare que todas as consultas seguem o mesmo molde da estrela: **uma fato + JOIN nas dimensões**.

In [ ]:
print("[Hive] Os 10 produtos que mais deram lucro:")
display(consultar_dw("""
    SELECT p.nome_produto, p.categoria,
           SUM(f.quantidade)            AS itens,
           ROUND(SUM(f.valor_total), 2) AS faturamento,
           ROUND(SUM(f.lucro), 2)       AS lucro,
           ROUND(100.0 * SUM(f.lucro) / SUM(f.valor_total), 1) AS margem_pct
    FROM fato_vendas f
    JOIN dim_produto p ON p.sk_produto = f.sk_produto
    GROUP BY p.nome_produto, p.categoria
    ORDER BY lucro DESC
    LIMIT 10
"""))

print("[Hive] Ticket médio por segmento e turno:")
display(consultar_dw("""
    SELECT a.segmento, a.turno,
           COUNT(DISTINCT f.id_venda) AS vendas,
           ROUND(SUM(f.valor_total) / COUNT(DISTINCT f.id_venda), 2) AS ticket_medio
    FROM fato_vendas f
    JOIN dim_aluno a ON a.sk_aluno = f.sk_aluno
    WHERE a.sk_aluno <> 0
    GROUP BY a.segmento, a.turno
    ORDER BY ticket_medio DESC
"""))

print("[Hive] Turmas com maior valor em atraso:")
display(consultar_dw("""
    SELECT a.turma,
           COUNT(*) AS dividas,
           ROUND(SUM(f.valor_devido), 2)   AS total_fiado,
           ROUND(SUM(f.valor_atrasado), 2) AS total_atrasado
    FROM fato_fiado f
    JOIN dim_aluno a ON a.sk_aluno = f.sk_aluno
    WHERE a.sk_aluno <> 0
    GROUP BY a.turma
    ORDER BY total_atrasado DESC
    LIMIT 10
"""))

print("[Hive] Quatro dimensões numa só pergunta — faturamento de cada categoria, à vista × fiado, "
      "em dias com e sem chuva:")
display(consultar_dw("""
    SELECT p.categoria, pg.tipo_pagamento, c.choveu,
           ROUND(SUM(f.valor_total), 2) AS faturamento
    FROM fato_vendas f
    JOIN dim_produto   p  ON p.sk_produto    = f.sk_produto
    JOIN dim_pagamento pg ON pg.sk_pagamento = f.sk_pagamento
    JOIN dim_clima     c  ON c.sk_clima      = f.sk_clima
    WHERE c.sk_clima <> 0
    GROUP BY p.categoria, pg.tipo_pagamento, c.choveu
    ORDER BY p.categoria, pg.tipo_pagamento, c.choveu
"""))

## 6. Regressão: quantos itens a cantina vai vender amanhã? (Aula 06)

**Problema real:** a cantina prepara salgados, lanches e sucos **antes** de saber quanto vai vender.
Se prepara demais, vira perda (seção 4.4); se prepara de menos, perde venda.

**Solução:** um modelo de **regressão** (que prevê um número) aprende com os dias anteriores
como o dia da semana, o clima e o calendário influenciam o volume de vendas.

- **Alvo (o que queremos prever):** `itens_vendidos` no dia.
- **Pistas (features):** dia da semana, dia do mês, temperatura média e máxima, chuva,
  condição do tempo, turno letivo e tipo de dia.
- **Modelo:** Random Forest — o mesmo da aula: várias "árvores de decisão" votando juntas.

Diferente da aula, separamos treino e teste **pela data** (treina com os dias mais antigos e
testa nos mais recentes). É assim que o modelo seria usado na vida real: prevendo o futuro.

In [ ]:
ALVO_REGRESSAO = "itens_vendidos"
num_reg = ["temperatura_media", "temperatura_max", "precipitacao_mm", "dia_do_mes"]
cat_reg = ["dia_semana", "condicao", "turno_letivo", "tipo_dia"]

base_reg = dias.sort_values("data").reset_index(drop=True)
X = base_reg[num_reg + cat_reg]
y = base_reg[ALVO_REGRESSAO]

# shuffle=False: os 20% de dias mais recentes ficam para o teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)
print(f"Treino: {len(X_train)} dias | Teste: {len(X_test)} dias "
      f"(a partir de {base_reg.loc[X_test.index[0], 'data']:%d/%m/%Y})")


def criar_preprocessador(numericas, categoricas):
    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())])
    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))])
    return ColumnTransformer(transformers=[
        ("num", numeric_transformer, numericas),
        ("cat", categorical_transformer, categoricas)])


modelo_reg = Pipeline(steps=[
    ("preprocessor", criar_preprocessador(num_reg, cat_reg)),
    ("regressor", RandomForestRegressor(n_estimators=300, min_samples_leaf=2, random_state=42))])
modelo_reg.fit(X_train, y_train)
y_pred = modelo_reg.predict(X_test)

# Comparação com um "chute" simples: prever sempre a média do treino
chute = np.full(len(y_test), y_train.mean())
metricas_reg = pd.DataFrame({
    "Modelo (Random Forest)": [mean_absolute_error(y_test, y_pred),
                               np.sqrt(mean_squared_error(y_test, y_pred)),
                               r2_score(y_test, y_pred)],
    "Chute (média)": [mean_absolute_error(y_test, chute),
                      np.sqrt(mean_squared_error(y_test, chute)),
                      r2_score(y_test, chute)]},
    index=["MAE (erro médio, em itens)", "RMSE", "R²"])
display(metricas_reg)
print(f"Em média, o modelo erra {metricas_reg.iloc[0, 0]:.0f} itens por dia "
      f"(o chute erraria {metricas_reg.iloc[0, 1]:.0f}). "
      f"Média de itens vendidos por dia: {y.mean():.0f}.")

In [ ]:
datas_teste = base_reg.loc[X_test.index, "data"].dt.strftime("%d/%m")
fig, ax = plt.subplots(figsize=(12, 4.5))
ax.plot(np.arange(len(datas_teste)), y_test.values, color=CORES[0], linewidth=2, marker="o", markersize=6, label="O que realmente vendeu")
ax.plot(np.arange(len(datas_teste)), y_pred, color=CORES[1], linewidth=2, marker="o", markersize=6, linestyle="--",
        label="O que o modelo previu")
ax.set_title("O modelo consegue prever as vendas do dia?")
ax.set_ylabel("itens vendidos no dia")
eixo_dias(ax, datas_teste)
ax.legend(loc="upper left", bbox_to_anchor=(0, -0.08), ncol=2)
plt.tight_layout()
plt.show()
como_ler("a linha **azul** é o que a cantina vendeu de verdade nos dias de teste e a **laranja tracejada** é o "
         "que o modelo previu, sem ter visto esses dias. Quanto mais próximas as duas linhas, melhor o modelo.")

NOMES_AMIGAVEIS = {"temperatura_media": "Temperatura média", "temperatura_max": "Temperatura máxima",
                   "precipitacao_mm": "Chuva (mm)", "dia_do_mes": "Dia do mês", "dia_semana": "Dia da semana",
                   "condicao": "Tempo", "turno_letivo": "Turno", "tipo_dia": "Tipo de dia"}


def nome_amigavel(nome):
    nome = nome.split("__", 1)[1]
    if nome in NOMES_AMIGAVEIS:
        return NOMES_AMIGAVEIS[nome]
    for coluna, rotulo in NOMES_AMIGAVEIS.items():
        if nome.startswith(coluna + "_"):
            return f"{rotulo}: {nome[len(coluna) + 1:].replace('_', ' ')}"
    return nome


# O que mais pesou na previsão
nomes = modelo_reg.named_steps["preprocessor"].get_feature_names_out()
importancia = (pd.Series(100 * modelo_reg.named_steps["regressor"].feature_importances_, index=nomes)
                 .rename(nome_amigavel).nlargest(10))
importancia = importancia[importancia >= 0.5]
fig, ax = plt.subplots(figsize=(10, 5))
barras(ax, importancia.index, importancia.values, COR, "O que mais ajuda a prever quanto a cantina vai vender?",
       "peso na previsão (%)", formato=lambda v: f"{br(v, 1)}%")
plt.tight_layout()
plt.show()
como_ler("quanto maior a barra, mais aquela informação ajudou o modelo a acertar. Todas as barras juntas "
         "(incluindo as que não aparecem) somam 100%.")

### 5.1 Usando o modelo: previsão e sugestão de produção

Informe como será o dia (pela previsão do tempo) e o modelo estima o total de itens.
Com a participação histórica de cada produto, sugerimos quanto preparar dos **perecíveis**.

In [ ]:
novo_dia = pd.DataFrame([{
    "temperatura_media": 27.0,
    "temperatura_max": 32.0,
    "precipitacao_mm": 0.0,
    "dia_do_mes": 3,
    "dia_semana": "Sex",
    "condicao": "ensolarado",
    "turno_letivo": "integral",
    "tipo_dia": "letivo",
}])
previsao_itens = modelo_reg.predict(novo_dia)[0]
print(f"Previsão para o dia informado: {previsao_itens:.0f} itens")

# Participação de cada produto, calculada só com dias de temperatura parecida (±3 °C)
parecidos = ve[(ve["temperatura_media"] - novo_dia.loc[0, "temperatura_media"]).abs() <= 3]
if parecidos.empty:
    parecidos = ve
participacao = parecidos.groupby(["categoria", "nome_produto"])["quantidade"].sum()
participacao = participacao / participacao.sum()
sugestao = (participacao * previsao_itens).round().astype(int).rename("sugestao_de_producao")
display(sugestao.sort_values(ascending=False).head(15).reset_index())

## 7. Classificação: esta compra fiado vai atrasar? (Aula 05)

**Problema real:** parte das vendas é "fiado" — o responsável paga depois. Algumas dívidas atrasam.

**Solução:** um modelo de **classificação** (que prevê uma categoria) aprende com as dívidas antigas
quais características aparecem mais nas que atrasaram.

- **Alvo:** `atrasou` = 1 se a dívida passou de **30 dias** sem pagamento (ou foi paga depois disso), 0 se foi paga em dia.
- **Pistas:** valor da dívida, quantas dívidas o aluno já tinha antes e quanto somavam, série, idade,
  segmento, turno, restrição alimentar, dia da semana e dia do mês.
- Usamos apenas dívidas com **mais de 30 dias**: as mais novas ainda não tiveram tempo de atrasar,
  então não sabemos o "final da história" delas.
- `class_weight="balanced"`: como atrasos são minoria, o modelo dá mais peso a eles para não ignorá-los.

In [ ]:
PRAZO_DIAS = 30

clf_base = dev.dropna(subset=["turma"]).sort_values(["ano_letivo", "id_aluno", "data_divida"]).copy()
# Histórico do aluno ATÉ aquela dívida (nunca usa informação do futuro)
clf_base["dividas_anteriores"] = clf_base.groupby(["ano_letivo", "id_aluno"]).cumcount()
clf_base["valor_anterior"] = (clf_base.groupby(["ano_letivo", "id_aluno"])["valor_devido"].cumsum()
                              - clf_base["valor_devido"])
clf_base["dia_semana"] = clf_base["data_divida"].dt.dayofweek.map(DIAS_SEMANA)
clf_base["dia_do_mes"] = clf_base["data_divida"].dt.day

data_ref = devedores["data_divida"].max()
clf_base = clf_base[clf_base["data_divida"] <= data_ref - pd.Timedelta(days=PRAZO_DIAS)]
dias_para_pagar = (clf_base["data_quitacao"] - clf_base["data_divida"]).dt.days
clf_base["atrasou"] = ((clf_base["status"] == STATUS_ATRASADO)
                       | (dias_para_pagar > PRAZO_DIAS)
                       | ((clf_base["status"] == STATUS_ABERTO))).astype(int)

num_clf = ["valor_devido", "dividas_anteriores", "valor_anterior", "serie", "idade", "dia_do_mes"]
cat_clf = ["segmento", "turno", "restricao_alimentar", "dia_semana"]
X = clf_base[num_clf + cat_clf]
y = clf_base["atrasou"]
print(f"{len(clf_base)} dívidas analisadas | {y.mean():.1%} atrasaram")

In [ ]:
# Para treinar e testar, cada grupo (pagou em dia / atrasou) precisa ter pelo menos 2 dívidas
CLASSIFICACAO_POSSIVEL = y.nunique() == 2 and y.value_counts().min() >= 2
if not CLASSIFICACAO_POSSIVEL:
    print("Ainda não há dívidas atrasadas E pagas em dia suficientes para treinar o modelo "
          f"(por grupo: {y.value_counts().to_dict()}). Esta seção roda sozinha quando houver mais dados.")
else:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    modelo_clf = Pipeline(steps=[
        ("preprocessor", criar_preprocessador(num_clf, cat_clf)),
        ("classifier", RandomForestClassifier(n_estimators=300, min_samples_leaf=3,
                                              class_weight="balanced", random_state=42))])
    modelo_clf.fit(X_train, y_train)
    y_pred = modelo_clf.predict(X_test)

    print(f"Acurácia: {accuracy_score(y_test, y_pred):.2%}  (acertos no geral)")
    print(f"Precisão: {precision_score(y_test, y_pred, average='macro'):.2%}  "
          "(quando o modelo diz 'vai atrasar', quanto acerta)")
    print(f"Recall:   {recall_score(y_test, y_pred, average='macro'):.2%}  "
          "(dos atrasos reais, quantos o modelo encontrou)")
    print(f"F1-Score: {f1_score(y_test, y_pred, average='macro'):.2%}  (equilíbrio entre os dois)")

    # Acertos e erros (a "matriz de confusão" desenhada como colunas)
    cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
    grupos = ["Dívidas que foram\npagas em dia", "Dívidas que\natrasaram"]
    acertos, erros = [cm[0, 0], cm[1, 1]], [cm[0, 1], cm[1, 0]]
    posicao = np.arange(len(grupos))
    fig, ax = plt.subplots(figsize=(9, 4.8))
    ax.bar(posicao - 0.19, acertos, 0.38, color=COR_OK, edgecolor="white", linewidth=2, label="O modelo acertou")
    ax.bar(posicao + 0.19, erros, 0.38, color=COR_ERRO, edgecolor="white", linewidth=2, label="O modelo errou")
    ax.set_xticks(posicao, grupos)
    ax.set_title("Quantas dívidas o modelo classificou certo?")
    ax.set_ylabel("número de dívidas (teste)")
    ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    ax.grid(axis="x", visible=False)
    escrever_valores(ax)
    ax.legend()
    plt.tight_layout()
    plt.show()
    como_ler("separamos dívidas que o modelo **nunca viu**. Em cada grupo, a coluna **verde** conta quantas ele "
             "acertou e a **vermelha**, quantas errou. Um bom modelo tem as verdes bem mais altas que as vermelhas.")

### 6.1 Usando o modelo: risco de uma nova compra fiado

In [ ]:
if CLASSIFICACAO_POSSIVEL:
    nova_divida = X.iloc[[0]].copy()
    nova_divida["valor_devido"] = 18.50
    nova_divida["dividas_anteriores"] = 6
    nova_divida["valor_anterior"] = 75.00

    print("Dados da nova compra fiado:")
    display(nova_divida.T.rename(columns={nova_divida.index[0]: "valor"}))
    probabilidade = modelo_clf.predict_proba(nova_divida)[0]
    for classe, prob in zip(["Pagar em dia", "Atrasar"], probabilidade):
        print(f"Chance de {classe}: {prob:.1%}")

    fig, ax = plt.subplots(figsize=(9, 2.6))
    barras(ax, ["Pagar em dia", "Atrasar"], 100 * probabilidade, [COR_OK, COR_ERRO],
           "Qual a chance desta compra fiado atrasar?", "chance (%)", formato=lambda v: f"{br(v, 0)}%")
    ax.set_xlim(0, 115)
    plt.tight_layout()
    plt.show()

## 8. Apache Spark: SQL e streaming (Aula 09)

O **Spark** processa dados distribuídos entre várias máquinas e também dados que **chegam
continuamente** (*Structured Streaming*).

Na cantina, as vendas chegam em **lotes** ao longo do dia (é exatamente o que o gerador de dados faz).
Vamos:
1. usar o **Spark SQL** para recalcular uma tabela Gold e conferir com o pandas;
2. simular a chegada das vendas **dia após dia** e ver um painel sendo atualizado sozinho.

In [ ]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

spark = SparkSession.builder.appName("CantinaEscolar").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
print("Spark iniciado. Versão:", spark.version)

colunas_spark = ["id_venda", "data", "hora", "turma", "categoria", "nome_produto",
                 "quantidade", "valor_total", "forma_pagamento"]
ve_spark = ve[colunas_spark].copy()
ve_spark["data"] = ve_spark["data"].dt.strftime("%Y-%m-%d")
ve_spark[["turma", "categoria", "nome_produto"]] = (ve_spark[["turma", "categoria", "nome_produto"]]
                                                    .fillna("Não identificado"))
sdf = spark.createDataFrame(ve_spark)
sdf.createOrReplaceTempView("vendas")

print("[Spark SQL] Faturamento por categoria e mês:")
spark.sql('''
    SELECT substr(data, 1, 7)          AS mes,
           categoria,
           SUM(quantidade)             AS itens,
           ROUND(SUM(valor_total), 2)  AS faturamento
    FROM vendas
    GROUP BY substr(data, 1, 7), categoria
    ORDER BY mes, faturamento DESC
''').show(15, truncate=False)

total_spark = sdf.agg(F.sum("valor_total")).first()[0]
print(f"Faturamento total — Spark: R$ {total_spark:,.2f} | pandas: R$ {ve['valor_total'].sum():,.2f}")

### 8.1 Structured Streaming: painel ao vivo

Cada "dia" de vendas é gravado como um novo arquivo JSON numa pasta. O Spark vigia essa pasta
(`readStream`) e, a cada arquivo novo, **atualiza sozinho** o faturamento por categoria
(`outputMode("complete")` = sempre a tabela inteira, já somada).

In [ ]:
N_DIAS_SIMULADOS = 10

PASTA_STREAM = "/content/stream_cantina"
PASTA_ENTRADA = f"{PASTA_STREAM}/entrada"
PASTA_TEMP = f"{PASTA_STREAM}/temporarios"
PASTA_CHECKPOINT = f"{PASTA_STREAM}/checkpoint"
shutil.rmtree(PASTA_STREAM, ignore_errors=True)
os.makedirs(PASTA_ENTRADA)
os.makedirs(PASTA_TEMP)

esquema = StructType([
    StructField("data", StringType()),
    StructField("hora", IntegerType()),
    StructField("categoria", StringType()),
    StructField("quantidade", IntegerType()),
    StructField("valor_total", DoubleType()),
])
fluxo = spark.readStream.schema(esquema).json(PASTA_ENTRADA)

painel = (fluxo.groupBy("categoria")
               .agg(F.sum("quantidade").alias("itens"),
                    F.round(F.sum("valor_total"), 2).alias("faturamento")))

consulta = (painel.writeStream
                  .format("memory")
                  .queryName("painel_ao_vivo")
                  .outputMode("complete")
                  .option("checkpointLocation", PASTA_CHECKPOINT)
                  .start())

dias_simulados = sorted(ve_spark["data"].unique())[:N_DIAS_SIMULADOS]
for numero, dia in enumerate(dias_simulados, start=1):
    lote = ve_spark.loc[ve_spark["data"] == dia, ["data", "hora", "categoria", "quantidade", "valor_total"]]
    arquivo = f"lote_{numero:03d}.json"
    # Escreve na pasta temporária e só depois move: o Spark nunca lê um arquivo pela metade
    lote.to_json(os.path.join(PASTA_TEMP, arquivo), orient="records", lines=True, force_ascii=False)
    os.replace(os.path.join(PASTA_TEMP, arquivo), os.path.join(PASTA_ENTRADA, arquivo))
    consulta.processAllAvailable()   # espera o Spark processar o lote que acabou de chegar
    acumulado = spark.sql("SELECT SUM(itens) AS itens, SUM(faturamento) AS fat FROM painel_ao_vivo").first()
    print(f"Lote {numero:2d} ({dia}) chegou → acumulado: {acumulado['itens']:>6,} itens | "
          f"R$ {acumulado['fat']:>10,.2f}")

print("\nPainel após todos os lotes:")
painel_final = spark.sql("SELECT * FROM painel_ao_vivo ORDER BY faturamento DESC").toPandas()
display(painel_final)

consulta.stop()

In [ ]:
pf = painel_final.sort_values("faturamento", ascending=False)
fig, ax = plt.subplots(figsize=(10, 4.5))
colunas(ax, pf["categoria"], pf["faturamento"], [COR_CATEGORIA.get(c, COR_NEUTRA) for c in pf["categoria"]],
        f"Quanto cada categoria vendeu nos {N_DIAS_SIMULADOS} dias do streaming?", "faturamento (R$)",
        formato=reais, rotacao=12)
eixo_reais(ax)
plt.tight_layout()
plt.show()
como_ler("resultado final do painel \"ao vivo\": a soma de tudo que chegou nos lotes simulados, por categoria.")
spark.stop()

## 9. Resumo dos achados e exportação

Os principais números, calculados automaticamente a partir dos dados.

In [ ]:
fat_total = ve["valor_total"].sum()
lucro_total = ve["lucro"].sum()
prejuizo_perdas = pe["valor_prejuizo"].sum()
top_produto = ve.groupby("nome_produto")["quantidade"].sum().idxmax()
dia_pico = (ve.groupby("dia_semana")["id_venda"].nunique() / n_dias_semana).idxmax()
hora_pico = ve.groupby("hora")["id_venda"].nunique().idxmax()
cat_perda = gold["desperdicio_categoria"].iloc[0]
clima_top = dias.groupby("condicao")["itens_vendidos"].mean()
pct_fiado_atrasado = 100 * dev["valor_atrasado"].sum() / dev["valor_devido"].sum()

resumo = [
    ("Data Warehouse (modelo estrela)", f"{len(FATOS)} fatos e {len(DIMENSOES)} dimensões; "
                                        f"fato_vendas com {len(fato_vendas):,} linhas"),
    ("Faturamento no período", f"R$ {fat_total:,.2f}"),
    ("Lucro bruto (venda - custo)", f"R$ {lucro_total:,.2f} ({100 * lucro_total / fat_total:.1f}%)"),
    ("Prejuízo com perdas", f"R$ {prejuizo_perdas:,.2f} ({100 * prejuizo_perdas / fat_total:.1f}% do faturamento)"),
    ("Produto mais vendido", top_produto),
    ("Dia da semana de pico", dia_pico),
    ("Horário de pico", f"{hora_pico}h"),
    ("Categoria com mais desperdício", f"{cat_perda['categoria']} (R$ {cat_perda['prejuizo']:,.2f})"),
    ("Tempo com mais vendas / menos vendas", f"{clima_top.idxmax()} / {clima_top.idxmin()}"),
    ("Fiado em atraso", f"{pct_fiado_atrasado:.1f}% do valor vendido fiado"),
    ("Erro médio da previsão de vendas", f"{metricas_reg.iloc[0, 0]:.0f} itens/dia "
                                         f"(média de {dias['itens_vendidos'].mean():.0f} itens/dia)"),
]
display(pd.DataFrame(resumo, columns=["Indicador", "Valor"]).set_index("Indicador"))

In [ ]:
# Salva cada tabela Gold e cada tabela do DW em CSV (o banco cantina_dw.db já foi salvo nas seções 3.2 e 5.2)
for nome, tabela in gold.items():
    para_sql(tabela).to_csv(os.path.join(PASTA_SAIDA, f"gold_{nome}.csv"), index=False, encoding="utf-8-sig")
for nome, tabela in {**DIMENSOES, **FATOS}.items():
    para_sql(tabela).to_csv(os.path.join(PASTA_SAIDA, f"dw_{nome}.csv"), index=False, encoding="utf-8-sig")
dw.close()

arquivo_zip = shutil.make_archive("/content/resultados_cantina", "zip", PASTA_SAIDA)
print("Arquivos gerados:", sorted(os.listdir(PASTA_SAIDA)))
print("Tudo compactado em:", arquivo_zip)

BAIXAR_RESULTADOS = False   # mude para True para baixar o .zip pelo navegador
if BAIXAR_RESULTADOS:
    from google.colab import files
    files.download(arquivo_zip)